# Proprietary Momentum Strategy — DJIA 30, Apr 2016 – Apr 2026

**MIT 15.C51 Spring 2026 — Project #1 (Proprietary Trading track, momentum half).**

This notebook develops, backtests, iteratively improves, and documents a proprietary cross-sectional momentum strategy on the Dow Jones 30, culminating in an Investment Committee Memorandum (Stage 7). Mean reversion is a separate deliverable and is explicitly out of scope.

---

### How to run

1. **Kernel → Restart & Run All.** Every cell is designed to execute top-to-bottom from a fresh kernel.
2. Dependencies are installed inline in the first code cell. Required libraries: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `yfinance`, `pandas_datareader`, `hmmlearn`, `statsmodels`, `xgboost`.
3. Data is fetched live (no cached files committed). Prices/volume via `yfinance`, risk-free rate via FRED `DTB3`, Fama-French factors via `pandas_datareader.famafrench`.
4. All tunable parameters live in the `RUN_CONFIG` dict in Stage 0 — do **not** introduce magic numbers elsewhere. Strategy functions read from `RUN_CONFIG` (passed as `cfg`).
5. Intermediate results are checkpointed to `results.pkl` after Stage 3 and Stage 5 so re-running is cheap.
6. Stage structure follows PROMPT.md §3 — each stage ends with a git commit tagged `stage-N: <what>`.

### Non-negotiables (baked into the pipeline)

- **No look-ahead.** Weights are always shifted by one trading day before being multiplied by realised returns.
- **Point-in-time universe.** The DJIA constituent map is reconstructed from `BASELINE_APR2016` + `CHANGES`; no survivorship bias.
- **Transaction costs.** Applied on portfolio turnover (`weights.diff().abs().sum(axis=1)`) at 10 bps base case. Sensitivities at 0/5/15/20 bps in Stage 6.
- **Walk-forward validation.** Expanding-window folds (see `RUN_CONFIG['WALK_FORWARD_FOLDS']`). No hyperparameter tuning on test windows.
- **Benchmarks.** Every strategy is compared against EW-DJIA buy-and-hold, `^DJI` (price-weighted), SPY, and compounded DTB3.

In [ ]:
# One-shot dependency install. Safe to re-run; pip is idempotent.
# hmmlearn / statsmodels / xgboost / pandas_datareader are not in Colab's default image.
!pip install --quiet hmmlearn statsmodels xgboost pandas_datareader

---

## Stage 0 — Setup & Reproducibility

Single source of truth for every parameter (`RUN_CONFIG`), pinned RNG seeds, version pins for the audit trail, and a global matplotlib style for publication-quality figures.

In [ ]:
from __future__ import annotations

import logging
import random
import sys
import warnings
from dataclasses import dataclass
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('momentum')

In [ ]:
# ── RUN_CONFIG ────────────────────────────────────────────────────────────
# Single source of truth. Every downstream cell reads from this dict; no
# magic numbers elsewhere in the notebook.

SEED = 42

RUN_CONFIG: dict[str, Any] = {
    # Universe / horizon
    'START_DATE':        '2016-04-18',
    'END_DATE':          '2026-04-18',
    'FFILL_LIMIT_DAYS':  10,

    # Portfolio construction
    'REBALANCE_FREQ':    'ME',           # pandas month-end alias
    'LONG_PCT':          0.20,           # long top quintile
    'SHORT_PCT':         0.20,           # short bottom quintile

    # Transaction costs (per unit turnover, one-way → charged on abs Δweight)
    'COST_BPS':          10,             # base case
    'COST_BPS_GRID':     [0, 5, 10, 15, 20],  # Stage 6 sensitivity

    # Signal lookbacks (trading days)
    'LOOKBACKS': {
        'mom_long':      252,            # 12-month total window
        'mom_mid':       126,            # 6-month
        'mom_short':     63,             # 3-month
        'skip':          21,             # skip-one-month lag
        'vol_long':      60,             # vol estimator
        'vol_short':     20,             # short-horizon vol for scaling
        'ma_fast':       50,
        'ma_slow':       200,
        'beta_window':   252,            # residual-momentum regression window
        'vol_target':    0.10,           # annualised target for Stage 5 vol-targeting
    },

    # Walk-forward expanding-window folds (train_end is inclusive, test_start exclusive of train_end)
    'WALK_FORWARD_FOLDS': [
        {'train_start': '2016-04-18', 'train_end': '2020-12-31',
         'test_start':  '2021-01-01', 'test_end':  '2021-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2021-12-31',
         'test_start':  '2022-01-01', 'test_end':  '2022-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2022-12-31',
         'test_start':  '2023-01-01', 'test_end':  '2026-04-18'},
    ],

    # HMM regime model (Stage 2 strategy #13)
    'HMM_N_STATES':      3,
    'HMM_EXPOSURE':      {'bull': 1.0, 'choppy': 0.5, 'bear': 0.0},

    # Evaluation
    'BOOTSTRAP_N':       1000,
    'BOOTSTRAP_BLOCK':   20,             # block length for stationary bootstrap
    'NEWEY_WEST_LAG':    5,

    # Reproducibility
    'SEED':              SEED,
}

random.seed(SEED)
np.random.seed(SEED)

log.info('RUN_CONFIG loaded: %d top-level keys, seed=%d',
         len(RUN_CONFIG), RUN_CONFIG['SEED'])

In [ ]:
# ── Version pins (audit trail) ────────────────────────────────────────────
# Printed once so the exact environment is recoverable from a saved run.

import importlib

_LIBS = [
    'pandas', 'numpy', 'sklearn', 'matplotlib', 'yfinance',
    'pandas_datareader', 'hmmlearn', 'statsmodels', 'xgboost', 'scipy',
]

print(f'Python           {sys.version.split()[0]}')
for name in _LIBS:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{name:<18} {ver}')
    except ImportError:
        print(f'{name:<18} NOT INSTALLED')

In [ ]:
# ── Global matplotlib style (set once, inherited by every figure) ─────────

mpl.rcParams.update({
    'figure.figsize':       (12, 5),
    'figure.dpi':           110,
    'savefig.dpi':          160,
    'savefig.bbox':         'tight',
    'font.family':          'DejaVu Sans',
    'font.size':            10,
    'axes.titlesize':       12,
    'axes.titleweight':     'semibold',
    'axes.labelsize':       10,
    'axes.grid':            True,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'axes.prop_cycle':      mpl.cycler(color=[
        '#0B3D91',   # deep navy
        '#C0392B',   # brick
        '#117A65',   # forest
        '#B9770E',   # burnt orange
        '#6C3483',   # plum
        '#7F8C8D',   # slate
    ]),
    'grid.alpha':           0.25,
    'grid.linestyle':       '--',
    'lines.linewidth':      1.6,
    'legend.frameon':       False,
    'legend.fontsize':      9,
})

log.info('matplotlib style configured')

---

## Stage 1 — Data Layer

Three data streams feed the entire project:

1. **DJIA constituents** — reconstructed day-by-day from an April-2016 baseline set plus the six index reconstitutions in the sample window, so no stock receives weight on a date it wasn't actually in the index (no survivorship bias).
2. **Prices & volume** — `yfinance` auto-adjusted daily (splits + cash distributions folded into the close), forward-filled up to `FFILL_LIMIT_DAYS` trading days for short gaps. One post-2024 delisting (`WBA`, taken private by Sycamore Partners in 2025) is dropped from the download universe; this reduces the effective membership by one name during the 2018-06-26 → 2024-02-26 window in which WBA was an index member.
3. **Risk-free rate** — FRED `DTB3` (3-month T-bill secondary-market yield, annualised %), converted to a daily rate via `(r/100)/252` and forward-filled to every business day.

Each stream is wrapped in a pure function driven by `RUN_CONFIG`, so Stage 1 can be re-run deterministically without touching downstream code.

In [ ]:
# ── DJIA 30 constituent reconstruction ──────────────────────────────────
# Baseline as of April 2016; explicit changes capture the six known index
# reconstitutions in the ten-year sample window. Effective dates and
# ticker replacements cross-checked against S&P DJI press releases.

BASELINE_APR2016: set[str] = {
    'AAPL', 'AXP', 'BA',  'CAT',  'CSCO', 'CVX',  'DD',
    'DIS',  'GE',  'GS',  'HD',   'IBM',  'INTC', 'JNJ',
    'JPM',  'KO',  'MCD', 'MMM',  'MRK',  'MSFT', 'NKE',
    'PFE',  'PG',  'RTX', 'TRV',  'UNH',  'V',    'VZ',
    'WMT',  'XOM',
}

# (effective_date, added_tickers, removed_tickers)
CHANGES: list[tuple[str, list[str], list[str]]] = [
    ('2018-06-26', ['WBA'],              ['GE']),
    ('2019-04-02', ['DOW'],              ['DD']),
    ('2020-04-06', ['RTX'],              ['UTX']),     # post-merger relabel (UTX→RTX)
    ('2020-08-31', ['AMGN','CRM','HON'], ['XOM','PFE','RTX']),
    ('2024-02-26', ['AMZN','SHW'],       ['WBA','INTC']),
    ('2024-11-01', ['NVDA'],             ['DOW']),
]

# WBA was taken private (Sycamore Partners, 2025) and no longer has a
# continuous yfinance history. Dropping it from the download universe
# costs one name on 2018-06-26 → 2024-02-26 but keeps the data pipeline
# deterministic. Documented in the ICM data-quality section.
EXCLUDE_FROM_DOWNLOAD: set[str] = {'WBA'}


def build_constituent_map(
    baseline: set[str],
    changes: list[tuple[str, list[str], list[str]]],
    start: str,
    end: str,
    exclude_from_download: set[str] = EXCLUDE_FROM_DOWNLOAD,
) -> tuple[pd.DataFrame, list[str]]:
    """Reconstruct day-by-day DJIA membership over a business-day index.

    Args:
        baseline: tickers in the index on the first date of the window.
        changes: ordered reconstitutions as (effective_date, added, removed).
        start, end: window endpoints (inclusive) in YYYY-MM-DD form.
        exclude_from_download: tickers to strip from the column set because
            price data is unavailable. They are still honored historically
            — the effective universe is just one name smaller during
            periods when they were members.

    Returns:
        (constituent_map, all_tickers) where constituent_map is a
        (business_day × ticker) boolean DataFrame and all_tickers is the
        sorted column list with excluded tickers removed.
    """
    days = pd.bdate_range(start, end)
    changes_sorted = sorted(changes, key=lambda x: pd.Timestamp(x[0]))

    current = set(baseline)
    change_idx = 0
    rows: list[tuple[pd.Timestamp, str]] = []

    for day in days:
        while change_idx < len(changes_sorted):
            eff_date = pd.Timestamp(changes_sorted[change_idx][0])
            if eff_date <= day:
                _, added, removed = changes_sorted[change_idx]
                current.update(added)
                current.difference_update(removed)
                change_idx += 1
            else:
                break
        for ticker in current:
            rows.append((day, ticker))

    df = pd.DataFrame(rows, columns=['date', 'ticker'])
    all_tickers = sorted(set(df['ticker']) - exclude_from_download)

    cmap = pd.DataFrame(False, index=days, columns=all_tickers)
    for date, group in df.groupby('date'):
        in_index = [t for t in group['ticker'] if t in all_tickers]
        cmap.loc[date, in_index] = True
    return cmap, all_tickers


cfg = RUN_CONFIG   # short alias used throughout
constituent_map, ALL_TICKERS = build_constituent_map(
    BASELINE_APR2016, CHANGES, cfg['START_DATE'], cfg['END_DATE']
)
log.info('Constituent map: %d business days × %d tickers (WBA excluded from download)',
         *constituent_map.shape)


In [ ]:
# ── yfinance and FRED wrappers ────────────────────────────────────────────

import yfinance as yf
import pandas_datareader.data as web


def download_prices_volume(
    tickers: list[str],
    start: str,
    end: str,
    ffill_limit: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Pull adjusted close + volume from yfinance for `tickers`.

    `auto_adjust=True` folds splits and cash distributions into the close,
    so the returned price panel is directly usable for return construction.
    Short gaps (≤ `ffill_limit` trading days) are carried forward to absorb
    the occasional missing tick; longer gaps are preserved as NaN and
    surface in the Stage 1 validation report.
    """
    raw = yf.download(tickers, start=start, end=end,
                      auto_adjust=True, progress=False)
    prices = raw['Close'].sort_index().ffill(limit=ffill_limit)
    volume = raw['Volume'].sort_index().ffill(limit=ffill_limit)
    return prices, volume


def load_risk_free(start: str, end: str) -> pd.Series:
    """Fetch FRED DTB3 and convert to a daily risk-free rate series.

    FRED reports DTB3 as an annualised percentage (secondary-market yield
    on the 3-month T-bill). We approximate the daily rate by `(r/100)/252`;
    the error versus true geometric compounding is < 1 bp at DTB3 levels
    observed in sample.
    """
    raw = web.DataReader('DTB3', 'fred', start, end).squeeze()
    daily = (raw / 100.0) / 252.0
    biz_days = pd.date_range(start, end, freq='B')
    return daily.reindex(biz_days).ffill()


In [ ]:
# ── Execute the data pipeline ─────────────────────────────────────────────
# prices, volume, rf_daily are module-level globals consumed by every
# downstream stage. Rerunning this cell is the single point of refresh.

log.info('Downloading %d tickers from yfinance (%s → %s)...',
         len(ALL_TICKERS), cfg['START_DATE'], cfg['END_DATE'])

prices, volume = download_prices_volume(
    ALL_TICKERS,
    cfg['START_DATE'],
    cfg['END_DATE'],
    cfg['FFILL_LIMIT_DAYS'],
)

rf_daily = load_risk_free(cfg['START_DATE'], cfg['END_DATE'])

# Align the constituent map to the actual trading-day index from yfinance
# (business days minus US holidays). Reindex preserves only dates for
# which we have price data; gaps between b-days collapse cleanly.
constituent_map = (
    constituent_map.reindex(prices.index).ffill().fillna(False).astype(bool)
)

log.info('Prices  : %s', prices.shape)
log.info('Volume  : %s', volume.shape)
log.info('RF rate : %s  avg %.2f%% annualised',
         rf_daily.shape, rf_daily.mean() * 252 * 100)
log.info('Const.  : %s  avg %.1f stocks/day',
         constituent_map.shape, constituent_map.sum(axis=1).mean())


In [ ]:
# ── Data validation ───────────────────────────────────────────────────────

def data_validation_report(
    prices: pd.DataFrame,
    volume: pd.DataFrame,
    constituent_map: pd.DataFrame,
) -> pd.DataFrame:
    """Per-ticker summary of coverage, gaps, and constituent tenure.

    Flags tickers whose largest consecutive missing-price run exceeds
    the ffill limit — candidates for exclusion and worth a note in the
    ICM data-quality section.
    """
    rows = []
    for t in prices.columns:
        px = prices[t]
        is_member = constituent_map.get(t, pd.Series(False, index=prices.index))
        member_days = int(is_member.sum())
        na_mask = px[is_member].isna()
        na_count = int(na_mask.sum())
        if na_mask.any():
            groups = (na_mask != na_mask.shift()).cumsum()
            max_gap = int(na_mask.groupby(groups).sum().max())
        else:
            max_gap = 0
        rows.append({
            'ticker':       t,
            'first_member': is_member.idxmax() if member_days else pd.NaT,
            'last_member':  is_member[::-1].idxmax() if member_days else pd.NaT,
            'member_days':  member_days,
            'na_in_tenure': na_count,
            'na_pct':       (na_count / member_days) if member_days else np.nan,
            'max_gap_days': max_gap,
        })
    out = pd.DataFrame(rows).set_index('ticker')
    return out.sort_values('max_gap_days', ascending=False)


report = data_validation_report(prices, volume, constituent_map)
print(f"Price panel  : {prices.shape[0]:>5} rows × {prices.shape[1]:>2} cols")
print(f"Date range   : {prices.index.min().date()} → {prices.index.max().date()}")
print(f"Avg #stocks  : {constituent_map.sum(axis=1).mean():.1f} per day")
print(f"Min #stocks  : {constituent_map.sum(axis=1).min()} on "
      f"{constituent_map.sum(axis=1).idxmin().date()}")
print(f"Max in-tenure gap : {report['max_gap_days'].max()} trading days "
      f"(ticker={report['max_gap_days'].idxmax()})")

dropped = report[report['max_gap_days'] > cfg['FFILL_LIMIT_DAYS']]
if len(dropped):
    print(f"\nTickers with in-tenure gaps > {cfg['FFILL_LIMIT_DAYS']}d (review):")
    print(dropped[['member_days', 'na_in_tenure', 'max_gap_days']])
else:
    print(f"\nAll in-tenure gaps ≤ {cfg['FFILL_LIMIT_DAYS']} trading days ✓")

report.head(15)


In [ ]:
# ── Universe composition over time ────────────────────────────────────────
# Left: rolling count of index members by day (reconstitutions show as
# step changes). Right: Gantt-style membership tenure per ticker — the
# six reconstitution events become easy to eyeball.

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.3]}
)

n_members = constituent_map.sum(axis=1)
ax1.plot(n_members.index, n_members.values, color='#0B3D91', linewidth=1.4)
ax1.axhline(30, color='gray', linestyle=':', linewidth=0.9, label='Target (30)')
ax1.fill_between(n_members.index, 28, 30, alpha=0.05, color='#0B3D91')
ax1.set_title('DJIA membership count over sample window')
ax1.set_ylabel('Number of constituents')
ax1.set_ylim(26, 31)
ax1.legend(loc='lower left')

sorted_tickers = (
    constituent_map.sum(axis=0).sort_values(ascending=True).index.tolist()
)
for i, t in enumerate(sorted_tickers):
    mask = constituent_map[t]
    if not mask.any():
        continue
    grp = (mask != mask.shift()).cumsum()
    for _, run in mask.groupby(grp):
        if run.iloc[0]:
            ax2.hlines(i, run.index.min(), run.index.max(),
                       colors='#0B3D91', linewidth=3.0, alpha=0.85)
ax2.set_yticks(range(len(sorted_tickers)))
ax2.set_yticklabels(sorted_tickers, fontsize=7)
ax2.set_title('Membership tenure by ticker')
ax2.set_xlim(pd.Timestamp(cfg['START_DATE']), pd.Timestamp(cfg['END_DATE']))
ax2.grid(axis='x', alpha=0.25, linestyle='--')
ax2.set_axisbelow(True)

plt.tight_layout()
plt.show()


---

## Stage 2 — Strategy Zoo (15 momentum variants)

Every strategy is a pure function with the signature
`strategy_name(prices, volume, constituent_map, cfg) -> weights_df`.
Weights are daily with monthly rebalancing (month-end snapshot
forward-filled to every trading day). The long leg is the top
`cfg['LONG_PCT']` quintile, short leg the bottom `cfg['SHORT_PCT']`
quintile, equal-weighted inside each; deviations (TSMOM, vol-scaled)
are flagged per strategy.

**Taxonomy.**

1. **Classical** (1-6) — canonical cross-sectional and time-series
   momentum from the academic literature: Jegadeesh & Titman (1993),
   Moskowitz/Ooi/Pedersen (2012), Antonacci (2012).
2. **Advanced** (7-12) — signal-quality improvements: risk-adjustment,
   residualisation (Blitz/Huij/Martens 2011), sector-neutrality,
   reversal overlays, acceleration, volume confirmation.
3. **Regime-aware / ML** (13-15) — HMM-gated exposure
   (Daniel & Moskowitz 2016 spirit), XGBoost stacking, rank ensemble.

**Invariants enforced by the shared helpers.**

- Non-constituents receive `NaN` signal and therefore zero weight on dates they weren't in the index.
- Dollar-neutral portfolios: long weights sum to +1, short to −1; total gross exposure ≤ 2.
- No look-ahead: `backtest()` shifts weights by one trading day before multiplying by realised returns, so signal at date *t* produces a trade at *t+1*.
- Transaction costs are charged on `|Δw|` at `cfg['COST_BPS']` (10 bps base case).

Stage 2 is delivered across multiple commits (≈ 3 strategies each per PROMPT.md §6 step 4). This commit lays down the shared infrastructure and the three classical cross-sectional-momentum variants.

In [ ]:
# ── Shared helpers consumed by every strategy ─────────────────────────────

def signal_to_weights(
    signal: pd.DataFrame,
    constituent_map: pd.DataFrame,
    long_pct: float,
    short_pct: float,
) -> pd.DataFrame:
    """Rank a cross-sectional signal into market-neutral long/short weights.

    Higher signal -> longer position. Equal-weighted inside each quintile.
    Non-constituents are masked to NaN before ranking so dropped names do
    not consume quintile slots.

    Args:
        signal:          (date x ticker) DataFrame. Higher = more long.
        constituent_map: (date x ticker) boolean membership mask.
        long_pct:        top fraction to go long (e.g., 0.20 = top quintile).
        short_pct:       bottom fraction to short.

    Returns:
        Weights DataFrame summing to ~0 cross-sectionally with |weights|
        summing to ~2 when both legs are populated.
    """
    mask = constituent_map.reindex_like(signal).fillna(False)
    sig = signal.where(mask)
    ranks = sig.rank(axis=1, pct=True)
    long_mask  = (ranks >= 1 - long_pct).astype(float)
    short_mask = (ranks <= short_pct).astype(float)
    n_long  = long_mask.sum(axis=1).replace(0, np.nan)
    n_short = short_mask.sum(axis=1).replace(0, np.nan)
    long_w  =  long_mask.div(n_long,  axis=0)
    short_w = -short_mask.div(n_short, axis=0)
    return (long_w + short_w).fillna(0)


def monthly_rebalance(weights_daily: pd.DataFrame, freq: str) -> pd.DataFrame:
    """Sample weights at `freq` (e.g. 'ME') and hold constant until next rebalance."""
    return (
        weights_daily.resample(freq).last()
        .reindex(weights_daily.index)
        .ffill()
        .fillna(0)
    )


def backtest(
    weights: pd.DataFrame,
    prices: pd.DataFrame,
    cost_bps: float,
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Compute gross/net daily P&L under no-look-ahead and turnover cost.

    Signal at day t produces trades at t+1: weights are shifted by one day
    before the return dot product. Cost at day t = |Δweights[t]| × bps/10000.
    Charging on raw `|Δw|` (rather than dollar notional) is the standard
    L/S equity-factor convention and matches the baseline notebook.
    """
    asset_rets = prices.pct_change()
    traded = weights.shift(1)
    gross = (traded * asset_rets).sum(axis=1, min_count=1).fillna(0)
    turnover = weights.diff().abs().sum(axis=1).fillna(0)
    cost = turnover * (cost_bps / 10000.0)
    net = gross - cost.reindex(gross.index).fillna(0)
    return gross, net, turnover


def sanity_check_weights(weights: pd.DataFrame, name: str) -> None:
    """Log warnings on market-neutrality or leverage violations. Non-raising."""
    row_sum = weights.sum(axis=1).abs()
    gross = weights.abs().sum(axis=1)
    if row_sum.max() > 0.01:
        log.warning('%s: row sums not dollar-neutral (max |sum(w)|=%.3f)',
                    name, row_sum.max())
    if gross.max() > 2.01:
        log.warning('%s: leverage exceeded (max sum|w|=%.3f)', name, gross.max())
    if weights.isna().any().any():
        log.warning('%s: weights contain NaN', name)


def quick_summary(net: pd.Series, turnover: pd.Series, name: str) -> dict:
    """One-line headline metrics for Stage 2 spot-checks."""
    if net.std() == 0 or len(net) == 0:
        return {'name': name, 'ann_ret': 0.0, 'ann_vol': 0.0,
                'sharpe': np.nan, 'turnover_yr': 0.0}
    excess = net - rf_daily.reindex(net.index).fillna(0)
    ann_ret = (1 + net).prod() ** (252 / len(net)) - 1
    ann_vol = net.std() * np.sqrt(252)
    sharpe  = excess.mean() / net.std() * np.sqrt(252)
    return {
        'name': name,
        'ann_ret': ann_ret,
        'ann_vol': ann_vol,
        'sharpe':  sharpe,
        'turnover_yr': turnover.mean() * 252,
    }


In [ ]:
# ── Strategies 1-3 · Classical cross-sectional momentum ───────────────────
# Canonical Jegadeesh & Titman (1993) formulation with a skip-one-month lag
# (21 trading days). The three variants use 12-, 6-, and 3-month look-back
# windows to test whether intermediate-horizon continuation dominates shorter
# reversal or longer reversion effects in the DJIA universe.


def strat_cs_mom_12_1(prices, volume, constituent_map, cfg):
    """#1  Classical 12-1 cross-sectional momentum (Jegadeesh & Titman 1993)."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_cs_mom_6_1(prices, volume, constituent_map, cfg):
    """#2  6-1 cross-sectional momentum — shorter look-back, higher turnover."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_mid']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_cs_mom_3_1(prices, volume, constituent_map, cfg):
    """#3  3-1 cross-sectional momentum — shortest classical horizon."""
    lb = cfg['LOOKBACKS']
    signal = prices.shift(lb['skip']) / prices.shift(lb['mom_short']) - 1
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check classical variants (full-sample, pre-walk-forward) ─────────
# Stage 3 will enforce expanding-window out-of-sample evaluation; this cell
# is a sanity run so any construction bug shows up before we've committed
# 12 more strategies on top.

_strategies_so_far = {
    'cs_mom_12_1': strat_cs_mom_12_1,
    'cs_mom_6_1':  strat_cs_mom_6_1,
    'cs_mom_3_1':  strat_cs_mom_3_1,
}

STRATEGIES: dict = globals().get('STRATEGIES', {})
RESULTS: dict = globals().get('RESULTS', {})

_rows = []
for _name, _fn in _strategies_so_far.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

_spot = pd.DataFrame(_rows).set_index('name')
_spot[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':      '{:.2%}',
    'ann_vol':      '{:.2%}',
    'sharpe':       '{:.2f}',
    'turnover_yr':  '{:.1f}x',
})


In [ ]:
# ── Runtime context attached to cfg for strategy functions ────────────────
# Underscore-prefixed keys are non-serializable runtime objects (they live
# only for the duration of the current Python kernel). Strategies that need
# the risk-free rate or a market index pull them from here rather than
# re-deriving them each call.

cfg['_rf_daily']        = rf_daily
cfg['_market_returns']  = prices.mean(axis=1).pct_change()        # EW-DJIA daily rets


def rf_over_lookback(rf_daily: pd.Series, window: int, skip: int) -> pd.Series:
    """Compounded risk-free return over the (window-skip) days ending `skip` days ago.

    Matches the look-back horizon used by 12-1 / 6-1 / 3-1 momentum
    signals — i.e., the return an investor would have earned by holding
    cash over the same window the momentum signal measures.
    """
    eff_days = window - skip
    r = (1 + rf_daily).rolling(eff_days).apply(np.prod, raw=True) - 1
    return r.shift(skip)


In [ ]:
# ── Strategies 4-6 · Time-series and dual-momentum variants ───────────────
# Unlike the cross-sectional strategies above, these impose an *absolute*
# filter: each stock's signal is benchmarked against the risk-free rate
# over the look-back window (Moskowitz/Ooi/Pedersen 2012; Antonacci 2012),
# so in broadly bearish regimes the portfolio can disengage instead of
# forcibly ranking the 'least-bad' loser as a long.


def strat_tsmom_abs(prices, volume, constituent_map, cfg):
    """#4  Time-series absolute momentum — long if 12-1 > rf, else zero.

    Moskowitz/Ooi/Pedersen (2012). Per-stock independent: each name
    qualifies for a long position only if its own 12-1 return beats the
    compounded risk-free rate over the same window. Equal-weighted
    across qualifying longs. The portfolio is long-only and de-risks
    to cash when no stock qualifies (bear markets).
    """
    lb = cfg['LOOKBACKS']
    stock_ret = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    rf_cum = rf_over_lookback(cfg['_rf_daily'], lb['mom_long'], lb['skip'])
    rf_cum = rf_cum.reindex(stock_ret.index, method='ffill')

    mask = constituent_map.reindex_like(stock_ret).fillna(False)
    qualifies = (stock_ret.sub(rf_cum, axis=0) > 0) & mask
    n_longs = qualifies.sum(axis=1).replace(0, np.nan)
    w_daily = qualifies.astype(float).div(n_longs, axis=0).fillna(0)
    return monthly_rebalance(w_daily, cfg['REBALANCE_FREQ'])


def strat_dual_momentum(prices, volume, constituent_map, cfg):
    """#5  Dual momentum — absolute filter layered on cross-sectional ranks.

    Antonacci (2012), adapted to an L/S framework. Start from CS 12-1
    quintile weights, then drop long names whose absolute 12-1 return
    failed to beat the risk-free rate, and drop short names whose
    absolute 12-1 return *did* beat it. Each surviving leg is
    re-normalised to ±1 when populated; either leg can empty out in
    extreme regimes, producing one-sided exposure.
    """
    lb = cfg['LOOKBACKS']
    stock_ret = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    rf_cum = rf_over_lookback(cfg['_rf_daily'], lb['mom_long'], lb['skip'])
    rf_cum = rf_cum.reindex(stock_ret.index, method='ffill')

    mask = constituent_map.reindex_like(stock_ret).fillna(False)
    sig = stock_ret.where(mask)
    ranks = sig.rank(axis=1, pct=True)
    long_raw  = (ranks >= 1 - cfg['LONG_PCT'])  & stock_ret.sub(rf_cum, axis=0).gt(0)
    short_raw = (ranks <= cfg['SHORT_PCT'])     & stock_ret.sub(rf_cum, axis=0).lt(0)

    n_long  = long_raw.sum(axis=1).replace(0, np.nan)
    n_short = short_raw.sum(axis=1).replace(0, np.nan)
    w = (long_raw.astype(float).div(n_long,  axis=0).fillna(0)
         - short_raw.astype(float).div(n_short, axis=0).fillna(0))
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_tsmom_vol_scaled(prices, volume, constituent_map, cfg):
    """#6  Volatility-scaled TSMOM — per-stock direction × (target/realised vol).

    Moskowitz/Ooi/Pedersen (2012). sign(12-1 − rf) per stock, then each
    position scaled by (target vol / 60-day realised vol) so every name
    contributes roughly the same ex-ante risk. Leverage is capped at
    3× per name; final portfolio is re-scaled so aggregate gross = 2
    when signals fire, preserving the direction mix.
    """
    lb = cfg['LOOKBACKS']
    rets = prices.pct_change()
    stock_ret = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    stock_vol = rets.rolling(lb['vol_long']).std() * np.sqrt(252)
    rf_cum = rf_over_lookback(cfg['_rf_daily'], lb['mom_long'], lb['skip'])
    rf_cum = rf_cum.reindex(stock_ret.index, method='ffill')

    mask = constituent_map.reindex_like(stock_ret).fillna(False)
    direction = np.sign(stock_ret.sub(rf_cum, axis=0)).where(mask, 0.0)
    scale = (lb['vol_target'] / stock_vol).clip(upper=3.0)
    raw = (direction * scale).fillna(0)
    gross = raw.abs().sum(axis=1).replace(0, np.nan)
    w_daily = raw.mul(2.0 / gross, axis=0).fillna(0)
    return monthly_rebalance(w_daily, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check strategies 4-6 ────────────────────────────────────────────

_new = {
    'tsmom_abs':        strat_tsmom_abs,
    'dual_momentum':    strat_dual_momentum,
    'tsmom_vol_scaled': strat_tsmom_vol_scaled,
}

_rows = []
for _name, _fn in _new.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)  # TSMOM is long-only → expect the neutrality warning
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

pd.DataFrame(_rows).set_index('name')[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':     '{:.2%}',
    'ann_vol':     '{:.2%}',
    'sharpe':      '{:.2f}',
    'turnover_yr': '{:.1f}x',
})


In [ ]:
# ── GICS sector map for industry-neutral construction ───────────────────
# Hand-curated; sector assignments follow S&P GICS classifications that
# have been stable for each name across the 2016-2026 sample. A couple
# of well-known reclassifications (DIS moved from Consumer Disc. to
# Comm. Services in 2018; V reclassified between IT and Financials over
# the sample) are small-enough noise for a 30-name universe — the
# approximation is noted in the ICM.

SECTOR_MAP: dict[str, str] = {
    # Technology
    'AAPL': 'Technology', 'CRM': 'Technology', 'CSCO': 'Technology',
    'IBM':  'Technology', 'INTC': 'Technology', 'MSFT': 'Technology',
    'NVDA': 'Technology',
    # Financials
    'AXP': 'Financials', 'GS': 'Financials', 'JPM': 'Financials',
    'TRV': 'Financials', 'V':  'Financials',
    # Healthcare
    'AMGN': 'Healthcare', 'JNJ': 'Healthcare', 'MRK': 'Healthcare',
    'PFE':  'Healthcare', 'UNH': 'Healthcare',
    # Consumer Discretionary
    'AMZN': 'Consumer Discretionary', 'HD':  'Consumer Discretionary',
    'MCD':  'Consumer Discretionary', 'NKE': 'Consumer Discretionary',
    # Industrials
    'BA':  'Industrials', 'CAT': 'Industrials', 'HON': 'Industrials',
    'MMM': 'Industrials', 'GE':  'Industrials', 'RTX': 'Industrials',
    # Consumer Staples
    'KO':  'Consumer Staples', 'PG': 'Consumer Staples', 'WMT': 'Consumer Staples',
    # Communication Services
    'VZ':  'Communication Services', 'DIS': 'Communication Services',
    # Energy
    'CVX': 'Energy', 'XOM': 'Energy',
    # Materials
    'DD':  'Materials', 'DOW': 'Materials', 'SHW': 'Materials',
}

cfg['_sector_map'] = SECTOR_MAP

_covered = set(SECTOR_MAP) & set(prices.columns)
_missing = set(prices.columns) - set(SECTOR_MAP)
log.info('Sector map covers %d / %d downloaded tickers; missing=%s',
         len(_covered), prices.shape[1], sorted(_missing))


In [ ]:
# ── Strategies 7-9 · Risk-adjusted, residual, industry-neutral momentum ───
# Signal-quality refinements that address three known weaknesses of raw
# cross-sectional momentum:
#   (7) volatility heterogeneity across names distorts rankings;
#   (8) momentum effects load on market beta (Blitz/Huij/Martens 2011);
#   (9) sector concentration drives signal, not idiosyncratic skill.


def strat_cs_mom_risk_adj(prices, volume, constituent_map, cfg):
    """#7  Risk-adjusted CS momentum — 12-1 return divided by 60-day vol.

    Scales each name's signal by its own volatility so low-vol and
    high-vol stocks are compared on an information-ratio-like basis
    rather than raw return. Sharpens rankings under heterogeneous-vol
    regimes.
    """
    lb = cfg['LOOKBACKS']
    rets = prices.pct_change()
    stock_ret = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    stock_vol = rets.rolling(lb['vol_long']).std() * np.sqrt(252)
    signal = stock_ret / stock_vol.replace(0, np.nan)
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_residual_momentum(prices, volume, constituent_map, cfg):
    """#8  Residual momentum (Blitz, Huij & Martens 2011).

    Strips the market-beta component out of the momentum signal by
    subtracting `beta × market_12-1_return`, leaving the idiosyncratic
    residual. BHM documented that residual momentum has a substantially
    lower volatility drag and fewer momentum-crash episodes than raw
    12-1. Betas are rolling-regression estimates over
    `cfg.LOOKBACKS.beta_window` trading days against the EW-DJIA market
    return proxy.
    """
    lb = cfg['LOOKBACKS']
    rets = prices.pct_change()
    mkt = cfg['_market_returns'].reindex(rets.index)
    cov_sm  = rets.rolling(lb['beta_window']).cov(mkt)
    var_mkt = mkt.rolling(lb['beta_window']).var()
    beta = cov_sm.div(var_mkt, axis=0)

    raw_mom = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    mkt_price = (1 + mkt).cumprod()
    mkt_mom = mkt_price.shift(lb['skip']) / mkt_price.shift(lb['mom_long']) - 1

    signal = raw_mom.sub(beta.mul(mkt_mom, axis=0), axis=0)
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_industry_neutral(prices, volume, constituent_map, cfg):
    """#9  Industry-neutralised momentum — within-sector z-scores then rank.

    Converts each name's raw 12-1 return into a z-score relative to its
    sector peers (cross-time, same-date), which the usual long/short
    quintile construction then ranks on. Sectors with a single DJIA
    representative collapse to zero by construction (no within-sector
    dispersion) and cannot take a position — a fair limitation in a
    30-name universe. Sector assignments follow `cfg['_sector_map']`.
    """
    lb = cfg['LOOKBACKS']
    raw_mom = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    sector_map = cfg['_sector_map']

    sectors = sorted(set(sector_map.values()))
    z = pd.DataFrame(index=raw_mom.index, columns=raw_mom.columns, dtype=float)
    for s in sectors:
        tickers = [t for t in raw_mom.columns if sector_map.get(t) == s]
        if len(tickers) < 2:
            continue
        sub = raw_mom[tickers]
        mean = sub.mean(axis=1)
        std  = sub.std(axis=1, ddof=0).replace(0, np.nan)
        z[tickers] = sub.sub(mean, axis=0).div(std, axis=0)

    w = signal_to_weights(z, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check strategies 7-9 ────────────────────────────────────────────

_new = {
    'cs_mom_risk_adj':   strat_cs_mom_risk_adj,
    'residual_momentum': strat_residual_momentum,
    'industry_neutral':  strat_industry_neutral,
}

_rows = []
for _name, _fn in _new.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

pd.DataFrame(_rows).set_index('name')[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':     '{:.2%}',
    'ann_vol':     '{:.2%}',
    'sharpe':      '{:.2f}',
    'turnover_yr': '{:.1f}x',
})


In [ ]:
# ── Strategies 10-12 · Reversal overlay, acceleration, volume-confirmed ───
# Three signal augmentations that layer additional price/volume behaviour
# onto the classical 12-1 base:
#   (10) short-horizon reversal (Jegadeesh 1990);
#   (11) acceleration — the derivative of the momentum signal;
#   (12) volume confirmation — higher conviction when recent volume spikes.


def strat_reversal_overlay(prices, volume, constituent_map, cfg):
    """#10  Long 12-1 winners, short 1-month winners (reversal + momentum).

    Jegadeesh (1990) showed short-horizon (1-month) returns mean-revert
    cross-sectionally. Combining that reversal signal with longer-horizon
    momentum produces a dual signal: `12-1_mom − 1m_mom`. High-signal
    names are longer-term winners that are also short-term contrarian
    picks; low-signal names are longer-term losers that have also just
    rallied (prime short candidates).
    """
    lb = cfg['LOOKBACKS']
    mom_long  = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    mom_short = prices / prices.shift(lb['skip']) - 1
    signal = mom_long - mom_short
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_acceleration(prices, volume, constituent_map, cfg):
    """#11  Acceleration — change in 12-1 momentum over the last quarter.

    Signal = current 12-1 momentum − 12-1 momentum measured three months
    ago. Captures names whose momentum profile is strengthening (positive
    acceleration, go long) versus weakening (negative, go short), adding
    a second-derivative view on top of the raw level.
    """
    lb = cfg['LOOKBACKS']
    mom_now  = prices.shift(lb['skip'])              / prices.shift(lb['mom_long'])              - 1
    mom_lag  = prices.shift(lb['skip'] + lb['mom_short']) / prices.shift(lb['mom_long'] + lb['mom_short']) - 1
    signal = mom_now - mom_lag
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


def strat_volume_confirmed(prices, volume, constituent_map, cfg):
    """#12  Volume-confirmed momentum — 12-1 signal amplified by volume spikes.

    For each name: signal = mom_12_1 × (1 + log(recent_volume / avg_volume)).
    Recent volume is the 21-day mean (most-recent month); baseline is the
    252-day mean. Volume spikes amplify conviction — a winner with above-
    average volume ranks higher, a loser with above-average volume ranks
    lower (stronger distribution signal).
    """
    lb = cfg['LOOKBACKS']
    mom = prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1
    vol_recent = volume.rolling(lb['skip']).mean()
    vol_base   = volume.rolling(lb['mom_long']).mean().replace(0, np.nan)
    vol_ratio = (vol_recent / vol_base).replace([np.inf, -np.inf], np.nan)
    amp = 1 + np.log(vol_ratio)
    signal = mom * amp
    w = signal_to_weights(signal, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check strategies 10-12 ──────────────────────────────────────────

_new = {
    'reversal_overlay':  strat_reversal_overlay,
    'acceleration':      strat_acceleration,
    'volume_confirmed':  strat_volume_confirmed,
}

_rows = []
for _name, _fn in _new.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

pd.DataFrame(_rows).set_index('name')[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':     '{:.2%}',
    'ann_vol':     '{:.2%}',
    'sharpe':      '{:.2f}',
    'turnover_yr': '{:.1f}x',
})


### Walk-forward inside the strategy function

Strategies 13-14 are parametric — they fit a model (HMM or XGBoost) to
historical data before issuing a signal. For these, "no look-ahead" means
*every prediction must come from a model fit on data strictly prior to
the prediction date*. We implement this inside the strategy itself by
iterating over `cfg['WALK_FORWARD_FOLDS']`: the model re-fits at the
start of each test window using an expanding training set. Dates outside
any test window receive zero weight — the strategy simply doesn't trade
them.

In [ ]:
# ── Strategy 13 · HMM regime-gated momentum ─────────────────────────────

def strat_hmm_regime(prices, volume, constituent_map, cfg):
    """HMM regime-gated CS 12-1 momentum (Daniel & Moskowitz 2016 inspiration).

    Three-state Gaussian HMM fitted on EW-DJIA daily returns inside each
    walk-forward fold. States are labelled by their mean return:

        lowest mean  → bear    (exposure multiplier 0×)
        middle mean  → choppy  (exposure multiplier 0.5×)
        highest mean → bull    (exposure multiplier 1×)

    Base signal is the classical CS 12-1 L/S portfolio (strategy #1);
    the HMM's per-day regime scale is applied multiplicatively. Dates
    before the first test window (2021-01-01) have no OOS regime read
    and therefore zero weight — the strategy is silent during the
    initial training-only period.
    """
    from hmmlearn.hmm import GaussianHMM

    base = strat_cs_mom_12_1(prices, volume, constituent_map, cfg)
    market = cfg['_market_returns'].fillna(0)
    exposure = cfg['HMM_EXPOSURE']

    scales = pd.Series(0.0, index=base.index, name='regime_scale')
    for fold in cfg['WALK_FORWARD_FOLDS']:
        train = market.loc[fold['train_start']:fold['train_end']]
        test  = market.loc[fold['test_start']:fold['test_end']]
        if len(train) < 100 or len(test) == 0:
            continue

        hmm = GaussianHMM(
            n_components=cfg['HMM_N_STATES'],
            covariance_type='diag',
            random_state=cfg['SEED'],
            n_iter=200,
        )
        hmm.fit(train.values.reshape(-1, 1))
        means = hmm.means_.flatten()
        order = np.argsort(means)                        # ascending: bear, choppy, bull
        state_to_scale = {order[0]:  exposure['bear'],
                          order[-1]: exposure['bull']}
        for s in order[1:-1]:
            state_to_scale[s] = exposure['choppy']

        states = hmm.predict(test.values.reshape(-1, 1))
        scales.loc[test.index] = [state_to_scale[s] for s in states]

    # Snap the regime scale to the rebalance cadence so HMM day-to-day
    # state flickers don't generate spurious turnover — the scale only
    # updates when the portfolio is already being rebalanced.
    scales = (
        scales.resample(cfg['REBALANCE_FREQ']).last()
              .reindex(base.index).ffill().fillna(0.0)
    )
    return base.mul(scales, axis=0)


In [ ]:
# ── Strategy 14 · XGBoost-stacked momentum ──────────────────────────────

def strat_ml_stacked(prices, volume, constituent_map, cfg):
    """Gradient-boosted stacker over eight momentum / trend / vol features.

    Feature matrix (per date × ticker):
        mom_12_1, mom_6_1, mom_3_1, vol_60, ma_cross,
        accel (Δmom_12_1 over a quarter),
        rev   (12-1 minus 1-month),
        vol_spike (21d mean vol / 252d mean vol).

    Target: 21-day forward return (matches the monthly rebalance horizon).
    Samples are taken every 21 trading days to avoid overlapping-label
    leakage. XGBoost refits once per walk-forward fold on the expanding
    training window; OOS predictions from each fold are concatenated
    and used as the daily signal (forward-filled between monthly
    observations). Dates with no fitted prediction receive zero weight.
    """
    from xgboost import XGBRegressor

    lb = cfg['LOOKBACKS']
    rets = prices.pct_change()

    features = {
        'mom_12_1':  prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1,
        'mom_6_1':   prices.shift(lb['skip']) / prices.shift(lb['mom_mid'])  - 1,
        'mom_3_1':   prices.shift(lb['skip']) / prices.shift(lb['mom_short'])- 1,
        'vol_60':    rets.rolling(lb['vol_long']).std() * np.sqrt(252),
        'ma_cross':  prices.rolling(lb['ma_fast']).mean()
                     / prices.rolling(lb['ma_slow']).mean() - 1,
        'accel':     ((prices.shift(lb['skip']) / prices.shift(lb['mom_long'])) -
                      (prices.shift(lb['skip'] + lb['mom_short']) /
                       prices.shift(lb['mom_long'] + lb['mom_short']))),
        'rev':       ((prices.shift(lb['skip']) / prices.shift(lb['mom_long']) - 1) -
                      (prices / prices.shift(lb['skip']) - 1)),
        'vol_spike': np.log((volume.rolling(lb['skip']).mean()
                             / volume.rolling(lb['mom_long']).mean()
                               .replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)),
    }
    X = pd.concat({k: v.stack() for k, v in features.items()}, axis=1)

    y = (prices.shift(-lb['skip']) / prices - 1).stack().rename('target')
    panel = X.join(y, how='inner').replace([np.inf, -np.inf], np.nan).dropna()

    # Non-overlapping monthly labels — subsample every 21 trading days
    monthly = prices.index[::lb['skip']]
    panel = panel[panel.index.get_level_values(0).isin(monthly)]

    preds = pd.Series(index=panel.index, dtype=float)
    feat_cols = list(features.keys())
    for fold in cfg['WALK_FORWARD_FOLDS']:
        t0, t1 = pd.Timestamp(fold['train_start']), pd.Timestamp(fold['train_end'])
        s0, s1 = pd.Timestamp(fold['test_start']),  pd.Timestamp(fold['test_end'])
        dates = panel.index.get_level_values(0)
        tr = (dates >= t0) & (dates <= t1)
        te = (dates >= s0) & (dates <= s1)
        if tr.sum() < 50 or te.sum() == 0:
            continue
        model = XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.8,
            random_state=cfg['SEED'], verbosity=0, n_jobs=-1,
        )
        model.fit(panel.loc[tr, feat_cols], panel.loc[tr, 'target'])
        preds.loc[panel.index[te]] = model.predict(panel.loc[te, feat_cols])

    signal_sparse = preds.unstack()
    signal_daily  = signal_sparse.reindex(prices.index).ffill()

    w = signal_to_weights(signal_daily, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Spot-check strategies 13-14 ─────────────────────────────────────────

_new = {'hmm_regime': strat_hmm_regime, 'ml_stacked': strat_ml_stacked}

_rows = []
for _name, _fn in _new.items():
    _w = _fn(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, _name)  # HMM pre-fold-1 window is zero-weight → expect 'dollar-neutral' passes, leverage may be below 2
    _gross, _net, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES[_name] = _fn
    RESULTS[_name] = {'weights': _w, 'gross': _gross, 'net': _net, 'turnover': _to}
    _rows.append(quick_summary(_net, _to, _name))

# Restrict headline metrics to the OOS test window (2021 onwards) since
# these strategies are deliberately zero pre-2021.
_first_test = pd.Timestamp(cfg['WALK_FORWARD_FOLDS'][0]['test_start'])
_rows_oos = []
for _name, _fn in _new.items():
    _net = RESULTS[_name]['net'].loc[_first_test:]
    _to  = RESULTS[_name]['turnover'].loc[_first_test:]
    _rows_oos.append(quick_summary(_net, _to, _name + '  [OOS only]'))

_out = pd.DataFrame(_rows + _rows_oos).set_index('name')
_out[['ann_ret','ann_vol','sharpe','turnover_yr']].style.format({
    'ann_ret':     '{:.2%}',
    'ann_vol':     '{:.2%}',
    'sharpe':      '{:.2f}',
    'turnover_yr': '{:.1f}x',
})


In [ ]:
# ── Strategy 15 · Ensemble of ranks ─────────────────────────────────────

def strat_ensemble_ranks(prices, volume, constituent_map, cfg):
    """Average cross-sectional percentile rank across five base signals:
    cs_mom_12_1, cs_mom_6_1, cs_mom_3_1, cs_mom_risk_adj, residual_momentum.

    Rationale: each signal captures a different aspect of momentum
    (different horizons, a vol-normalised variant, a market-residualised
    variant). Averaging their ranks stabilises the composite against any
    one signal's noise, at the cost of diluting whichever is the
    strongest alpha-generator.
    """
    lb = cfg['LOOKBACKS']
    rets = prices.pct_change()
    mkt = cfg['_market_returns'].reindex(rets.index)

    cs_12_1 = prices.shift(lb['skip']) / prices.shift(lb['mom_long'])  - 1
    cs_6_1  = prices.shift(lb['skip']) / prices.shift(lb['mom_mid'])   - 1
    cs_3_1  = prices.shift(lb['skip']) / prices.shift(lb['mom_short']) - 1
    vol_60  = rets.rolling(lb['vol_long']).std() * np.sqrt(252)
    risk_adj = cs_12_1 / vol_60.replace(0, np.nan)

    cov_sm  = rets.rolling(lb['beta_window']).cov(mkt)
    var_mkt = mkt.rolling(lb['beta_window']).var()
    beta = cov_sm.div(var_mkt, axis=0)
    mkt_price = (1 + mkt).cumprod()
    mkt_mom = mkt_price.shift(lb['skip']) / mkt_price.shift(lb['mom_long']) - 1
    residual = cs_12_1.sub(beta.mul(mkt_mom, axis=0), axis=0)

    mask = constituent_map.reindex_like(cs_12_1).fillna(False)
    signals = [cs_12_1, cs_6_1, cs_3_1, risk_adj, residual]
    ranks = [s.where(mask).rank(axis=1, pct=True) for s in signals]
    avg_rank = sum(ranks) / len(ranks)

    w = signal_to_weights(avg_rank, constituent_map, cfg['LONG_PCT'], cfg['SHORT_PCT'])
    return monthly_rebalance(w, cfg['REBALANCE_FREQ'])


In [ ]:
# ── Master verification: all 15 strategies registered & summarised ────────

EXPECTED_STRATEGIES = [
    # Classical (1-6)
    'cs_mom_12_1', 'cs_mom_6_1', 'cs_mom_3_1',
    'tsmom_abs', 'dual_momentum', 'tsmom_vol_scaled',
    # Advanced (7-12)
    'cs_mom_risk_adj', 'residual_momentum', 'industry_neutral',
    'reversal_overlay', 'acceleration', 'volume_confirmed',
    # Regime-aware / ML (13-15)
    'hmm_regime', 'ml_stacked', 'ensemble_ranks',
]

# Register + run the ensemble (last strategy added in stage-2f)
if 'ensemble_ranks' not in STRATEGIES:
    _w = strat_ensemble_ranks(prices, volume, constituent_map, cfg)
    sanity_check_weights(_w, 'ensemble_ranks')
    _g, _n, _to = backtest(_w, prices, cfg['COST_BPS'])
    STRATEGIES['ensemble_ranks'] = strat_ensemble_ranks
    RESULTS['ensemble_ranks']    = {'weights': _w, 'gross': _g, 'net': _n, 'turnover': _to}

missing = [s for s in EXPECTED_STRATEGIES if s not in STRATEGIES]
assert not missing, f'Missing strategies: {missing}'
log.info('All %d strategies registered.', len(EXPECTED_STRATEGIES))

# Pre-walk-forward headline summary. The *real* leaderboard happens in
# Stage 3 under strict expanding-window evaluation; this table is just a
# sanity glance at what we've built.
stage2_summary = pd.DataFrame([
    quick_summary(RESULTS[n]['net'], RESULTS[n]['turnover'], n)
    for n in EXPECTED_STRATEGIES
]).set_index('name')

stage2_summary['sharpe_rank'] = stage2_summary['sharpe'].rank(ascending=False).astype(int)
stage2_summary = stage2_summary.sort_values('sharpe', ascending=False)

print(f'Strategies: {len(STRATEGIES)} registered')
print(f'Results   : {len(RESULTS)} cached weight/return tuples')
print(f'Full-sample Sharpe range: {stage2_summary["sharpe"].min():.2f}  to  {stage2_summary["sharpe"].max():.2f}')

stage2_summary[['ann_ret','ann_vol','sharpe','turnover_yr','sharpe_rank']].style.format({
    'ann_ret':     '{:.2%}',
    'ann_vol':     '{:.2%}',
    'sharpe':      '{:.2f}',
    'turnover_yr': '{:.1f}x',
    'sharpe_rank': '{:.0f}',
}).background_gradient(subset=['sharpe'], cmap='RdYlGn')


Stage 2 complete — 15 strategy functions are module-level globals, each
registered in `STRATEGIES` and cached in `RESULTS`. The headline table
above is *full-sample* and therefore contaminated with forward-looking
information for the two ML strategies; Stage 3 rebuilds the leaderboard
under strict walk-forward evaluation and is the only ranking that
ultimately matters. The top candidates by full-sample Sharpe will
typically — but not necessarily — also top the Stage 3 OOS leaderboard.

---

## Stage 3 — Evaluation Framework & Leaderboard

_Single `evaluate()` producing risk-adjusted metrics, FF3 regression, bootstrap Sharpe CI, Newey-West t-stats. Master leaderboard sorted by walk-forward OOS Sharpe. Implemented in the next stage commit._

---

## Stage 4 — Diagnostic Visuals (Top 5)

_Cumulative curves, rolling Sharpe, underwater, return distribution, monthly heatmap, FF3 exposure, turnover. Top 5 by OOS Sharpe only. Implemented in the next stage commit._

---

## Stage 5 — Iterative Improvement Loop

_Up to five rounds of diagnose → propose → implement → compare → decide, starting from the #1 ranked strategy. Early-stop when OOS Sharpe gain < 0.05. Implemented in the next stage commit._

---

## Stage 6 — Robustness Tests (Winner)

_Cost/frequency/lookback sensitivities, subperiod stability, Monte Carlo bootstrap, stress-period P&L, capacity estimate. Implemented in the next stage commit._

---

## Stage 7 — Investment Committee Memorandum

_Executive summary, strategy rationale, P&L conditions, statistical properties, 10-year performance review, risk factors, recommendation, Appendix A (source attribution incl. verbatim PROMPT.md and `llm_interactions.log`), Appendix B (iteration log). Written in the next stage commit._